## Library

In [2]:
import time
from dataretrieval import nwis
import pandas as pd
from IPython.display import clear_output

ERROR! Session/line number was not unique in database. History logging moved to new session 4158


## Download Gage Height (00065) for SWOT Version D

In [ ]:
# ============================================================================
# Download USGS gage height data for SWOT observation period
# ============================================================================
from dataretrieval import nwis
import pandas as pd
import time
from IPython.display import clear_output

# SWOT observation period (Version D): from operational start to end of October 2025
START, END = "2023-01-01", "2025-10-31"

STATES = [
    "AL","AK","AZ","AR","CA","CO","CT","DC","DE","FL","GA","HI","IA","ID","IL",
    "IN","KS","KY","LA","MA","MD","ME","MI","MN","MO","MS","MT","NC","ND","NE",
    "NH","NJ","NM","NV","NY","OH","OK","OR","PA","PR","RI","SC","SD","TN","TX",
    "UT","VA","VI","VT","WA","WI","WV","WY"
]

all_parts = []

for st in STATES:
    clear_output(wait=True)
    print(f"Processing {st}...")
    
    try:
        # Download gage height (00065)
        df = nwis.get_record(
            service="dv",
            stateCd=st,
            start=START, 
            end=END,
            parameterCd="00065",
            siteStatus="all",
        )
        
        if not df.empty:
            df["stateCd"] = st
            all_parts.append(df)
            print(f"  → {st}: {len(df)} records downloaded")
        else:
            print(f"  → {st}: No data")
            
    except Exception as e:
        print(f"  → {st}: Error - {str(e)}")
        continue
    
    time.sleep(0.5)

# Concatenate all state data
all_df = pd.concat(all_parts, axis=0, ignore_index=False)
print(f"\nTotal records: {len(all_df)}")

# Reset index to convert MultiIndex to columns (fix parquet issue)
all_df_reset = all_df.reset_index()

# Check site count
if 'site_no' in all_df_reset.columns:
    print(f"Total sites: {all_df_reset['site_no'].nunique()}")
else:
    print(f"Columns: {all_df_reset.columns.tolist()}")

# Convert any problematic object columns to string
for col in all_df_reset.columns:
    if all_df_reset[col].dtype == 'object':
        all_df_reset[col] = all_df_reset[col].astype(str)

# Save to parquet
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_gages/usgs_gage_height_23_25.parquet"
all_df_reset.to_parquet(output_path, index=False)
print(f"\nSaved to: {output_path}")

Processing WY...
  → WY: 1732 records downloaded

Total records: 4421282
